## 6 Итоговая оценка

**Входы**

- Выбранная модель, порог и test
- Метрики и правила оценивания из постановки задачи

**Выходы**

- Значения заданных метрик, графики и разбор ошибок

**Что делаем**

Рассчитаем на test метрики, заданные в этапе 1.2, и разберём ошибки. Модель и порог на этом шаге уже не меняем

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### Оценка по заданным критериям

Рассчитываем AP, ROC-AUC, Brier, Accuracy, Precision, Recall, F1, F2 и Balanced accuracy по определениям этапа 1.2. Сравниваем выбранную модель с Dummy на одной тестовой выборке

Матрица ошибок показывает TP, FP, FN и TN при зафиксированном пороге. В строках — фактический исход, в столбцах — прогноз: TN и FP в первой строке, FN и TP во второй

ROC-кривая сопоставляет полноту $\frac{TP}{TP+FN}$ с долей ложных тревог среди кредитов без дефолта $\frac{FP}{FP+TN}$. PR-кривая — точность $\frac{TP}{TP+FP}$ с полнотой $\frac{TP}{TP+FN}$ при разных порогах

**Калибровка** — согласованность прогнозируемых вероятностей с наблюдаемыми частотами дефолта. На test её только оцениваем, без донастройки

Чтобы понять разброс метрик, используем **bootstrap**: 300 раз собираем тестовый набор случайным выбором строк с повторами. Число кредитов каждого класса сохраняем; модель и порог не меняем. Показываем границы, между которыми лежат средние 95% полученных значений

Это ориентир для этой модели и этого test. Он не учитывает новое обучение, возможные повторы клиентов или изменение данных со временем

In [ ]:
# dev-only: загрузчик Python-кода; в сборке заменяется содержимым файла
exec(compile(__import__('pathlib').Path('src/notebooks/_code/evaluate.py').read_text(encoding='utf-8'), 'src/notebooks/_code/evaluate.py', 'exec'), globals())

#### Что ограничивает выводы

Мы проверили модель на одном случайном разбиении Kaggle-датасета. Не знаем время заявок, период наблюдения дефолта и то, встречается ли один человек несколько раз. Также неизвестно, насколько эти данные похожи на заявки конкретного банка. Редкие грейды представлены небольшим числом записей

Грейд и ставка должны быть известны до прогноза. Если нужна оценка до их назначения, состав признаков придётся пересмотреть и провести новый эксперимент

Для практического применения нужны проверки на более поздних и внешних данных, оценка стоимости ошибок и качества для разных групп заёмщиков. Полученные метрики сами по себе не обосновывают автоматический отказ в кредите